In [ ]:
"""
GENERATIVE AI CHATBOT FOR TRAVEL RECOMMENDATIONS
Uses Groq API (FREE & FAST)

Setup:
1. Get free API key from: https://console.groq.com
2. Set environment variable: export GROQ_API_KEY="your_key_here"
   OR create a .env file with: GROQ_API_KEY=your_key_here
"""

import os
from groq import Groq
import pandas as pd
import numpy as np
import pickle
import json

class TravelChatbot:
    def __init__(self, api_key=None):
        """Initialize chatbot with Groq API"""
        self.api_key = api_key or os.getenv('GROQ_API_KEY')
        
        if not self.api_key:
            raise ValueError("⚠️  GROQ_API_KEY not found! Get free key from: https://console.groq.com")
        
        self.client = Groq(api_key=self.api_key)
        self.model = "llama-3.3-70b-versatile"  # Fast and free model
        
        # Load ML model and data
        self.load_model_and_data()
        
        # System prompt for travel expert
        self.system_prompt = """You are an expert AI Travel Assistant for India. 
You provide personalized travel recommendations based on user preferences.
You are friendly, knowledgeable, and helpful. You always respond in a conversational tone.
When users ask for recommendations, consider: budget, duration, interests, companions, season, and accessibility.
Format your responses clearly with emojis for better readability.
Always be honest if you don't know something."""
    
    def load_model_and_data(self):
        """Load trained model and processed data"""
        try:
            with open('models/best_model.pkl', 'rb') as f:
                self.ml_model = pickle.load(f)
            
            with open('models/label_encoders.pkl', 'rb') as f:
                self.encoders = pickle.load(f)
            
            with open('models/scaler.pkl', 'rb') as f:
                self.scaler = pickle.load(f)
            
            self.df = pd.read_csv('data/processed_dataset.csv')
            print("✅ Models and data loaded successfully!")
            
        except FileNotFoundError as e:
            print(f"⚠️  Error loading files: {e}")
            print("Make sure you've run the preprocessing and training scripts first!")
    
    def get_travel_data_context(self, user_query):
        """Get relevant travel data based on user query"""
        # Simple keyword extraction for context
        keywords = user_query.lower()
        
        # Get top destinations by rating
        top_places = self.df.groupby('place_name').agg({
            'rating_given': 'mean',
            'place_type': 'first',
            'state': 'first',
            'budget_thousand_inr': 'mean',
            'safety_index': 'mean'
        }).sort_values('rating_given', ascending=False).head(5)
        
        context = f"""
Available Travel Data:
- Total destinations in database: {self.df['place_name'].nunique()}
- Top rated destinations: {', '.join(top_places.index.tolist())}
- Average budget range: ₹{self.df['budget_thousand_inr'].min():.0f}k - ₹{self.df['budget_thousand_inr'].max():.0f}k per day
- Popular seasons: {', '.join(self.df['season'].value_counts().head(3).index.tolist())}
"""
        return context
    
    def chat(self, user_message, conversation_history=None):
        """
        Send message to AI and get response
        
        Args:
            user_message: User's question/message
            conversation_history: List of previous messages (optional)
        
        Returns:
            AI response string
        """
        if conversation_history is None:
            conversation_history = []
        
        # Get relevant data context
        data_context = self.get_travel_data_context(user_message)
        
        # Build messages for API
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "system", "content": data_context}
        ]
        
        # Add conversation history
        messages.extend(conversation_history)
        
        # Add current user message
        messages.append({"role": "user", "content": user_message})
        
        try:
            # Call Groq API
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.7,
                max_tokens=1024,
                top_p=0.9
            )
            
            ai_response = response.choices[0].message.content
            return ai_response
            
        except Exception as e:
            return f"❌ Error: {str(e)}\nPlease check your API key and internet connection."
    
    def get_ml_recommendation(self, budget, duration, place_type, companions, interest):
        """Get ML-based recommendation"""
        # This is a simplified version - you'd expand this with actual prediction
        matching_places = self.df[
            (self.df['budget_thousand_inr'] <= budget) &
            (self.df['trip_duration_days'] <= duration) &
            (self.df['place_type'] == place_type)
        ].groupby('place_name').agg({
            'rating_given': 'mean',
            'state': 'first',
            'safety_index': 'mean'
        }).sort_values('rating_given', ascending=False).head(3)
        
        return matching_places

def interactive_chat():
    """Run interactive chatbot in terminal"""
    print("="*70)
    print("🌏 WELCOME TO AI TRAVEL ASSISTANT!")
    print("="*70)
    print("\nInitializing chatbot...")
    
    try:
        bot = TravelChatbot()
    except ValueError as e:
        print(f"\n{e}")
        print("\n📝 Setup Instructions:")
        print("1. Visit: https://console.groq.com")
        print("2. Sign up for free account")
        print("3. Generate API key")
        print("4. Set environment variable:")
        print("   export GROQ_API_KEY='your_key_here'")
        return
    
    print("\n✅ Chatbot ready! Ask me anything about travel in India!")
    print("💡 Examples:")
    print("   - 'Suggest best beach destinations for family'")
    print("   - 'I want to visit hill stations in December'")
    print("   - 'What are budget-friendly places for solo travel?'")
    print("\n📝 Type 'quit' or 'exit' to end conversation\n")
    print("="*70 + "\n")
    
    conversation_history = []
    
    while True:
        user_input = input("You: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\n👋 Thank you for using AI Travel Assistant! Happy travels!\n")
            break
        
        # Get AI response
        print("\n🤖 AI Assistant: ", end="", flush=True)
        response = bot.chat(user_input, conversation_history)
        print(response + "\n")
        
        # Update conversation history
        conversation_history.append({"role": "user", "content": user_input})
        conversation_history.append({"role": "assistant", "content": response})
        
        # Keep only last 6 messages (3 exchanges) to stay within context limits
        if len(conversation_history) > 6:
            conversation_history = conversation_history[-6:]

if __name__ == "__main__":
    # Check if running in interactive mode
    interactive_chat()

🌏 WELCOME TO AI TRAVEL ASSISTANT!

Initializing chatbot...


C:\Users\Mohammad Huzaifa\anaconda3\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator KNeighborsClassifier from version 1.5.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Mohammad Huzaifa\anaconda3\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.5.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Mohammad Huzaifa\anaconda3\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using versi

✅ Models and data loaded successfully!

✅ Chatbot ready! Ask me anything about travel in India!
💡 Examples:
   - 'Suggest best beach destinations for family'
   - 'I want to visit hill stations in December'
   - 'What are budget-friendly places for solo travel?'

📝 Type 'quit' or 'exit' to end conversation




You:  we want to visit manali in december 



🤖 AI Assistant: 🏔️ Manali in December can be a wonderful experience, especially if you enjoy winter sports and scenic landscapes covered in snow 🌨️. Here are a few things to keep in mind:

**Weather:** 🥶 It can get quite cold in Manali during December, with temperatures ranging from -2°C to 10°C. Make sure to pack warm clothing, including gloves, hats, and scarves.

**Activities:** 🎿 You can enjoy skiing, snowboarding, and other winter sports in the Solang Valley, which is about 14 km from Manali. You can also take a snow trek to nearby villages or go ice skating.

**Accessibility:** 🚗 The roads to Manali can be slippery and narrow during winter, so be prepared for a slow and careful drive. You can also take a bus or hire a taxi, but be aware that the journey may take longer than usual.

**Budget:** 💸 The average budget for a day in Manali can range from ₹5,000 to ₹20,000, depending on your accommodation and activities. You can find affordable hotels and guesthouses, but prices may be

You:  total 5 people and budget is 25000



🤖 AI Assistant: 👥 With 5 people and a budget of ₹25,000, I'd recommend planning a 4-day trip to Manali in December 📆. Here's a rough breakdown of how you can allocate your budget:

**Accommodation:** 🏨 You can find a decent hotel or guesthouse in Manali for ₹2,000-₹3,000 per night for a group of 5. For 4 nights, that would be around ₹8,000-₹12,000. Let's assume an average cost of ₹10,000 for accommodation.

**Food:** 🍴 You can find affordable restaurants and cafes in Manali, with meals starting from ₹100-₹200 per person. For 4 days, you can allocate around ₹5,000-₹6,000 for food, assuming ₹200-₹250 per person per day.

**Transportation:** 🚗 If you're traveling from a nearby city, you can take a bus or hire a taxi. For a group of 5, a taxi would be more convenient and would cost around ₹2,000-₹3,000 one way, depending on the distance. Let's assume a total transportation cost of ₹4,000-₹6,000 for the round trip.

**Activities:** 🎿 You can allocate around ₹2,000-₹3,000 per person for act

You:  very good , thank you , and tell us our budget is sufficient or not



🤖 AI Assistant: 🤔 Based on the estimates I provided earlier, your budget of ₹25,000 for 5 people for a 4-day trip to Manali in December seems a bit **tight** 📉. Here's why:

* Accommodation: ₹10,000 (average cost for 4 nights)
* Food: ₹5,500 (average cost for 4 days)
* Transportation: ₹5,000 (estimated cost for round trip)
* Activities: ₹4,500 (considering alternative activities)

Total: ₹25,000

As you can see, your budget is exactly equal to the estimated costs, leaving no room for any **unexpected expenses** 🤯 or **upgrades** 🚀. Additionally, the estimates I provided are rough and can vary depending on various factors like the specific accommodation you choose, the restaurants you eat at, and the activities you participate in.

Considering the peak season and the cold weather, I would recommend having a **buffer** of at least ₹5,000-₹10,000 to account for any unexpected expenses or changes in plans 📈.

So, to answer your question, your budget of ₹25,000 is **sufficient** 📊, but it'